In [ ]:
# capstone project -- function 4 (4D), week N

import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial import Delaunay, ConvexHull  # convex-hull check on the proposal

from bayes_tools import (
    fitting,
    normalize, initial_bounds, validate_bounds_consistency,
    generate_next_point, exploit_acquisition, ucb_acquisition,
    append_observations, compute_iteration_diagnostics,
    fit_gp, get_length_scales, loo_predictions,
    compare_kappa_proposals, print_kappa_comparison,
    backtest_acquisitions, print_backtest_summary,
)
from viz_tools import (
    plot_nd_slices, plot_loo_calibration, plot_kappa_sensitivity,
    plot_acquisition_backtest,
    plot_convergence, plot_acquisition_decay, plot_uncertainty_shrinkage,
    plot_step_distance,
)

# Function 4

4D, 33 observations collected (30 initial + weeks 2–4), **32 used for modelling** — the week-2 point is excluded from the fit. **Acquisition is `exploit`**: pure posterior-mean maximisation, no exploration term.

Proposal this week: **`[0.402322, 0.404984, 0.373926, 0.428738]`** (GP mean +0.3514, std 0.1399, predicted improvement **+0.0828**) — a new best if it holds.

## The week-2 point is excluded from the fit

It stays in the record; it is not used for modelling. It was a **poorly chosen submission**, not a property of the function: the old notebook built bounds without `lower_limit=0.0` and proposed an extrapolated corner with two negative coordinates. Submitted with the signs flipped, it returned **−215.0**, 6.6× worse than anything else.

It had to come out because one observation was reshaping the whole surrogate:

| | x0 | x1 | x2 | x3 | y std |
|---|---|---|---|---|---|
| excluded (live fit) | 0.860 | 0.853 | 0.890 | 0.803 | 8.00 |
| included | 1.849 | 1.986 | 1.621 | 1.690 | 34.99 |
| inflation | 2.15× | 2.33× | 1.82× | 2.11× | 4.4× |

That over-smoothing suppressed posterior σ exactly where the search needed resolution, and it had stalled `exploit` completely: step 0.0021 from the incumbent with a predicted gain of +0.0011. With the point excluded the same acquisition steps **0.0333** and predicts **+0.0828** — 16× the step, 75× the gain.

Two side effects worth knowing. Axis coverage was 49% / 50% / **98%** / **96%** — x2 and x3 were stretched almost entirely by that one corner; it is now a uniform 48–49%, and the convex hull went from 11.5% to **27.0%** of the bounding box. And `xi` is meaningful again: sized against the full record's spread of 215 it was 0.005%, against the modelling spread of 32.9 it is 0.030%.

**The risk this creates.** The surrogate no longer knows that corner is catastrophic, so nothing in the *model* stops the search returning there. The **convex-hull check in the proposal cell is the guard** — excluding the point shrinks the hull, so a corner proposal fails it. Do not weaken it. (The box check reports the corner as outside too, but only by 4e-7, which is a floating-point accident rather than protection.)

## Why `exploit`

`exploit` and low-`kappa` UCB are near-identical recommendations here, and everything with real exploration weight leaves the data cloud:

| kappa | proposal | in hull | pred. mean |
|---|---|---|---|
| 0.5 | `[0.406, 0.413, 0.360, 0.431]` | yes | +0.332 |
| 1.0 | `[0.411, 0.418, 0.341, 0.432]` | yes | +0.265 |
| 1.5 | `[0.417, 0.422, 0.320, 0.433]` | yes | +0.139 |
| **2.0** | `[1.933, 1.833, 1.836, 1.917]` | **no** | **−37.49** |
| 3.0, 5.0 | same corner | **no** | −37.49 |

`exploit`'s own proposal sits 0.037 from the `kappa=1` alternative, so choosing between them is close to a formality — but note the **mode switch has moved to between `kappa=1.5` and `kappa=2`**, down from between 3 and 4 before the exclusion. If `exploit` is ever swapped out, `kappa <= 1.5`.

EI and PI remain unusable: every `xi` from 0.5 to 20 returned the *identical* corner, because the improvement term is hopeless everywhere so EI collapses onto its `sigma*phi(z)` term and silently becomes `max_variance`. No `xi`, in any unit system, fixes that.

The backtest agrees, for what that is worth: `exploit` 2.056, `ucb_k1` 2.93, then `ucb_k2` 7.22 and everything else worse, with `max_var` last at 23.16. It is an exploitation-biased metric, so it was always going to rank `exploit` first — corroboration, not the reason.

## What to watch

- **`exploit` has no exploration term, so it can stall.** The proposal cell warns when the step falls below 0.01; that warning fired last week and the exclusion cleared it. If it fires again with the outlier already excluded, that is a real local optimum and the fallback is `ucb` at `kappa <= 1.5`.
- **The positive region is narrow and was missed entirely by 30 initial points.** The two positive observations sit 0.042 apart and 0.258 from the nearest initial point, which returned −4.03. So `y` falls by ~4.3 over that distance — steep, and worth remembering before trusting the GP far from the cluster.
- **All four axes are mildly relevant** (no pinned length-scales), so unlike functions 2 and 3 there is no flat subspace to restrict the search to.
- **No y-scaling.** `exploit` ranks by posterior mean and is exactly scale-invariant.
- LOO calibration: 32/32 points inside their own 95% interval.
- `plot_2d_bo` doesn't apply at D=4 — the GP view is `plot_nd_slices`. `plot_bo_diagnostics`/`plot_sample_trajectory` hard-code a 2D panel and are skipped.


In [2]:
X_initial = np.load("initial_data/function_4/initial_inputs.npy")
y_initial = np.load("initial_data/function_4/initial_outputs.npy")
n_initial = len(y_initial)
D = X_initial.shape[1]

assert D == 4, f"Expected a 4D problem, got {D}D input -- check the loaded file."

# Once, at the very start of the capstone:
new_X = np.empty((0, D))
new_y = np.empty((0,))

## Update this once per week

Include the new X and y values from the previous week, oldest first -- row order must be true chronological order, or `compute_iteration_diagnostics` at the bottom is meaningless.

One observation is recorded. See the header for two caveats attached to it: the sign of x0/x1 differs from what the old notebook actually proposed, and it is an extrapolation about twice as far out as any observed value on x2/x3.

In [ ]:
# Append last week's result BEFORE proposing this week's point, e.g.:
#
# new_X, new_y = append_observations(new_X, new_y, x_next, the_result_you_got)

new_X = np.array([
    [0.909972, 0.907092, 1.836169, 1.917449],
    [0.409027, 0.394466, 0.397297, 0.384784],
    [0.393738, 0.377868, 0.388294, 0.41894 ],
])

new_y = np.array([
    -215.0011203231484,
    0.15730397729049672,
    0.2685882660865464,
])

# ---------------------------------------------------------------------------
# EXCLUDED FROM THE GP FIT: the week-2 observation (index 0 above).
#
# It stays in the record -- it is a real measurement and the chronological order
# must not be disturbed -- but it is not used for modelling.
#
# WHY IT IS AN OUTLIER: it was a poorly chosen submission, not a property of the
# function. The old version of this notebook built bounds without
# lower_limit=0.0 and proposed [-0.909972, -0.907092, 1.836169, 1.917449] -- two
# physically impossible negative coordinates, and an extrapolated corner far
# outside the data cloud. Submitted with the signs flipped positive, it returned
# -215.0: 6.6x worse than anything else on record. It measures a region we have
# since decided never to search.
#
# WHY IT HAS TO COME OUT OF THE FIT rather than just be noted: it more than
# DOUBLES every ARD length-scale, so the GP believes the function is about twice
# as smooth as the other 32 points say.
#
#     length-scales with it    [1.85, 1.99, 1.62, 1.69]   y std 35.0
#     length-scales without it [0.86, 0.85, 0.89, 0.80]   y std  8.0
#
# That over-smoothing suppresses posterior sigma exactly where the search now
# needs resolution, and it is why `exploit` collapsed to re-proposing the
# incumbent (step 0.0021, predicted gain +0.0011). Drop it and the same
# acquisition steps 0.0333 and predicts +0.35.
#
# CONSEQUENCE TO KEEP IN VIEW: the surrogate no longer knows that corner is
# catastrophic. The convex-hull check in the proposal cell is now the thing
# stopping the search going back -- dropping this point shrinks the hull, so a
# corner proposal fails that test. Do not weaken it.
EXCLUDE_FROM_FIT = [0]        # indices into new_X / new_y

print("observations collected so far:", len(new_y),
      f"| excluded from the fit: {EXCLUDE_FROM_FIT} -> {len(new_y) - len(EXCLUDE_FROM_FIT)} used")
print("week 2 y: %.4f (%.1fx the previous worst, %.4f)"
      % (new_y[0], new_y[0] / y_initial.min(), y_initial.min()))
print("week 3 y: %.6f | best before it: %.4f" % (new_y[-1], y_initial.max()))
print("-> %s" % ("the week-3 point is the BEST observation on record, and the"
                 " first positive y seen"
                 if new_y[-1] > y_initial.max() else "within the existing range"))
print("-> retreating to the data cloud paid off: the y spread is now %.3f,"
      " still dominated by the week-2 outlier" % (max(new_y.max(), y_initial.max())
                                                  - min(new_y.min(), y_initial.min())))


## Build the full dataset (initial + everything collected so far)

In [ ]:
# X_all / y_all: the complete record, for reporting only.
# X / y: the MODELLING set, with EXCLUDE_FROM_FIT removed. Everything downstream
# -- GP fits, acquisition, bounds, hull, diagnostics -- uses X / y.
X_all, y_all = append_observations(X_initial, y_initial, new_X, new_y)

_use = np.ones(len(new_y), bool)
_use[EXCLUDE_FROM_FIT] = False
X, y = append_observations(X_initial, y_initial, new_X[_use], new_y[_use])

print("X_initial shape:", X_initial.shape, "| y_initial shape:", y_initial.shape)
print("full record      :", X_all.shape, "| modelling set:", X.shape)
for i in EXCLUDE_FROM_FIT:
    print(f"  excluded: {np.round(new_X[i], 6)}  y = {new_y[i]:.4f}"
          f"   (see the comment in the data cell)")
print(f"y range: full record [{y_all.min():.4f}, {y_all.max():.4f}]"
      f" | modelling [{y.min():.4f}, {y.max():.4f}]")
print(f"y std:   full record {y_all.std():.3f} | modelling {y.std():.3f}")

print("\nX range per dimension (modelling set):")
print("  min:", np.round(X.min(axis=0), 4))
print("  max:", np.round(X.max(axis=0), 4))
print("y spread: %.3f | std: %.3f" % (y.max() - y.min(), y.std()))

# X is known to never be negative -- lower_limit=0.0 is mandatory. Without it
# this function's padded bounds start near -0.91, and the old notebook duly
# proposed a point with two negative coordinates.
bounds = initial_bounds(X_initial, pad_fraction=1.0, lower_limit=0.0)

# Widen only if the MODELLING data needs it. With the week-2 corner excluded
# nothing should overshoot any more -- that point was the only reason this fix
# existed (its 6-dp x3 rounded ~4e-7 outside the padded bound). Kept because it
# is harmless and would be needed again if EXCLUDE_FROM_FIT is ever emptied.
overshoot = np.maximum(X.max(axis=0) - bounds[:, 1], 0.0)
if overshoot.any():
    print("\nwidening upper bounds to enclose observed data, by:", overshoot)
    bounds[:, 1] = np.maximum(bounds[:, 1], X.max(axis=0))
bounds[:, 0] = np.maximum(np.minimum(bounds[:, 0], X.min(axis=0)), 0.0)

validate_bounds_consistency(X, bounds)
print("\nBounds:\n", np.round(bounds, 4))

print("\nfraction of each bounds axis actually covered by data:")
for d in range(D):
    span = bounds[d, 1] - bounds[d, 0]
    print(f"  x{d}: {(X[:, d].max() - X[:, d].min()) / span:.1%}")

# The bounding box is a loose stand-in for "where the data is". At 4D with this
# few points the cloud is nearly all surface, so quantify the gap.
hull = Delaunay(X)
box = np.column_stack([X.min(axis=0), X.max(axis=0)])
box_volume = float(np.prod(box[:, 1] - box[:, 0]))
try:
    ch = ConvexHull(X)
    print(f"\nobserved bounding box volume: {box_volume:.4g}"
          f" | convex hull volume: {ch.volume:.4g} ({ch.volume / box_volume:.1%} of the box)")
    print(f"{len(ch.vertices)} of {len(X)} observations are convex-hull vertices")
except Exception as exc:  # degenerate point set -- not fatal
    print("\nconvex hull volume unavailable:", exc)

print(f"\nxi=0.01 would be {0.01 / (y.max() - y.min()):.4%} of the modelling y spread"
      f" ({y.max() - y.min():.3f})."
      "\nUCB needs no such correction: kappa is dimensionless.")
print("(the full record's spread is %.1f -- dominated by the excluded outlier,"
      " which is\nprecisely why any xi sized against it was meaningless)"
      % (y_all.max() - y_all.min()))

# The excluded point must stay OUT of reach: the GP no longer knows it is bad.
for i in EXCLUDE_FROM_FIT:
    inside_box = bool(np.all(new_X[i] >= bounds[:, 0]) and np.all(new_X[i] <= bounds[:, 1]))
    in_hull_excl = bool(hull.find_simplex(new_X[i]) >= 0)
    print(f"\nexcluded point {np.round(new_X[i], 4)}:")
    print(f"  still inside the search box: {inside_box}"
          + ("   <- the box alone will NOT keep the search out" if inside_box else ""))
    print(f"  inside the modelling hull:   {in_hull_excl}"
          + ("   <- the hull check WILL keep the search out" if not in_hull_excl else ""))

## How much of the GP fit rests on the one outlier

Worth quantifying before trusting any proposal. Refit on the initial batch alone and compare kernels: a single observation should not reshape the model, and here it does. The inflated length-scales and amplitude are what push variance-seeking acquisitions into the far field, so this cell is the explanation for the `kappa` choice below.

In [ ]:
# The evidence for excluding the week-2 point, recomputed rather than quoted.
# gp_check is the live surrogate (modelling set); gp_with refits including the
# excluded point, purely to show what it does to the kernel.
with fitting("live fit + a refit including the excluded point"):
    gp_check = fit_gp(X, y, bounds, n_restarts_optimizer=20, random_state=0)
    gp_with = fit_gp(X_all, y_all, bounds, n_restarts_optimizer=20, random_state=0)

print("LIVE fit, outlier excluded :", gp_check.kernel_)
print("refit including the outlier:", gp_with.kernel_)

ls_live = get_length_scales(gp_check)
ls_with = get_length_scales(gp_with)
print("\nlength-scales, excluded :", np.round(ls_live, 3))
print("length-scales, included :", np.round(ls_with, 3))
print("inflation from that ONE point:", np.round(ls_with / ls_live, 2))
print(f"y std: excluded {y.std():.2f} | included {y_all.std():.2f}")
print("-> a single observation more than doubling every length-scale is why it")
print("   is out of the fit. See the comment in the data cell.")

PINNED = 10.0  # fit_gp's default length_scale_bounds upper limit
pinned = [d for d, v in enumerate(ls_live) if v >= 0.999 * PINNED]
print("\npinned dimensions (GP treats as irrelevant):", pinned or "none")
print("-> all four axes are mildly relevant, so there is no flat subspace to")
print("   restrict the search to (unlike functions 2 and 3).")

## Backtest acquisition functions (using only data already collected)

A weak, free check on acquisition family: repeatedly split the data into a "seed" set (fits the GP) and a held-out "candidate" set (true y known, hidden from the fit), then see which config would have picked the best candidate most often. No new evaluations spent.

`xi` values here are in **raw `y`-units** (1.0 and 5.0, i.e. 0.5% and 2.4% of the 211-wide spread), since this notebook does not rescale `y`. The default `xi=0.01` would be meaningless at this magnitude.

**Read it as an exploitation-phase check only.** The metric scores recognition of points whose true `y` is already known, which rewards whatever ranks closest to the GP posterior mean and gives no credit for reducing uncertainty -- so `exploit` tends to win by construction and `max_variance` to lose. Expect `ucb_k5` to score poorly here too, which happens to agree with the separate finding below that `kappa=5` sends the search into the far field. That agreement is a coincidence of direction, not corroboration: this metric would penalise `kappa=5` regardless.

LIMITATION: it cannot test which config is best at proposing a genuinely new location, since there's no ground truth for that without a real evaluation.

In [ ]:
backtest_configs = [
    {"name": "ucb_k1",  "acquisition": "ucb", "kappa": 1.0},
    {"name": "ucb_k2",  "acquisition": "ucb", "kappa": 2.0},
    {"name": "ucb_k5",  "acquisition": "ucb", "kappa": 5.0},
    {"name": "ei_xi1",  "acquisition": "ei",  "xi": 1.0},
    {"name": "ei_xi5",  "acquisition": "ei",  "xi": 5.0},
    {"name": "pi_xi1",  "acquisition": "pi",  "xi": 1.0},
    {"name": "exploit", "acquisition": "exploit"},
    {"name": "max_var", "acquisition": "max_variance"},
]

with fitting("acquisition backtest, 50 splits"):
    backtest_results = backtest_acquisitions(
        X, y, bounds, backtest_configs,
        n_repeats=50, seed_frac=0.5, maximize=True,
        gp_kwargs={"n_restarts_optimizer": 15}, random_state=0,
    )

print_backtest_summary(backtest_results)

plot_acquisition_backtest(backtest_results)
plt.show()

## The kappa comparison: kept as the record of the alternative

`exploit` takes no hyperparameter, so this cell no longer chooses anything. It's kept for two reasons: it documents the mode switch that rules out high-exploration UCB, and it keeps `ucb` with a small `kappa` -- the conservative fallback to `exploit` -- measured rather than assumed.

What to look for is not the usual "where do proposals stop moving" but a **discontinuity**. Expect proposals to be almost identical for `kappa` from 0.5 to 3 (clustered near [0.40, 0.38, 0.39, 0.40], inside the convex hull, predicted mean about -2.64) and then to jump to a bounds corner at `kappa >= 4`, with predicted means of -131 to -160.

That jump is the outlier-inflated far-field variance winning out over a sensible posterior mean. The project default of `kappa=5.0` sits on the far side of it. Note how close the whole `kappa <= 3` band is to `exploit`'s own proposal -- that closeness is what makes the choice between them nearly a formality this week.

Re-check every week. The length-scales *have* since come back down -- the week-2 outlier is now excluded from the fit -- and the mode switch duly moved: it is between `kappa=1` and `kappa=2` now, not `kappa=3` and `kappa=4`. So `kappa <= 1` is the usable range if `exploit` is ever swapped out.

In [ ]:
# Both fits happen here, before any printing, so their ConvergenceWarnings
# cannot land in the middle of the tables below.
with fitting("kappa sweep + the ucb k=1 alternative"):
    kappa_rows = compare_kappa_proposals(X, y, bounds,
                                         kappa_values=[0.5, 1.0, 1.5, 2.0, 3.0, 5.0],
                                         maximize=True, n_restarts=30, random_state=0)
    alt, _ = generate_next_point(X, y, bounds, acquisition="ucb", kappa=1.0,
                                 maximize=True, n_restarts=40, random_state=0)

print_kappa_comparison(kappa_rows)

# Flag the discontinuity explicitly rather than leaving it to be eyeballed.
print("\nper-kappa: is the proposal inside the convex hull of the data?")
for row in kappa_rows:
    inside = bool(hull.find_simplex(row["x_next"]) >= 0)
    print(f"  kappa={row['kappa']:<5g} in_hull={str(inside):<5s} pred_mean={row['pred_mean']:+10.3f}")

plot_kappa_sensitivity(X, y, bounds, gp_check, kappa_rows, ucb_acquisition, maximize=True)
plt.show()

# --- The committed choice for this function ------------------------------
# exploit = pure posterior-mean maximisation; takes no hyperparameter.
# Conservative fallback, if convergence stalls or the surface looks multi-modal:
#     ACQ, ACQ_KWARGS = "ucb", {"kappa": 1.0}
ACQ = "exploit"
ACQ_KWARGS = {}

print(f"\nusing acquisition = {ACQ!r} {ACQ_KWARGS}")
print("  no hyperparameter, and exactly scale-invariant (ranks by posterior mean),")
print("  so no y-scaling is required.")

# How far is exploit's choice from the low-kappa UCB alternative? (fitted above)
print(f"\nucb kappa=1 would propose: {np.round(alt, 6)}")

## Propose the next point

Two checks run on the result. The **per-axis** check asks whether each coordinate is within the range already observed on its own axis. The **convex-hull** check is strictly stronger and is the one that matters: a point can pass the per-axis test and still sit outside the data cloud entirely, which is what every `kappa >= 4` proposal does.

Unlike function 3, the bounds here are the project-standard padded ones rather than the observed box -- no restriction is needed, because at `kappa=2` the proposal already lands inside the hull on its own.

In [ ]:
with fitting("the committed proposal"):
    x_next, gp = generate_next_point(
        X, y, bounds,
        acquisition=ACQ,  # see the comparison above; "exploit" takes no hyperparameter
        maximize=True,
        n_restarts=40,
        random_state=0,
        **ACQ_KWARGS,
    )

print(f"\n--- Next point to evaluate (bounds shape {bounds.shape}) ---")
print("x_next:", np.round(x_next, 6))
print(gp.kernel_)

mu, sigma = gp.predict(normalize(x_next.reshape(1, -1), bounds), return_std=True)
print(f"GP predicted mean: {mu[0]:.4g}, predicted std: {sigma[0]:.4g}")
print(f"current best observed y: {y.max():.4g}"
      f"  -> predicted improvement: {mu[0] - y.max():+.4g}")

print(f"\ndistance from the ucb kappa=1 alternative: {np.linalg.norm(x_next - alt):.4f}")

# --- Safety checks -------------------------------------------------------
on_edge = [d for d in range(D)
           if np.isclose(x_next[d], bounds[d, 0]) or np.isclose(x_next[d], bounds[d, 1])]
outside = [d for d in range(D) if not (X[:, d].min() <= x_next[d] <= X[:, d].max())]
print("\ndimensions where x_next sits on a bound:", on_edge or "none")
print("dimensions where x_next is outside the observed data range:", outside or "none")

in_hull = bool(hull.find_simplex(x_next) >= 0)
print(f"x_next inside the CONVEX HULL of the observations: {in_hull}")
if not in_hull:
    print("*** x_next is outside the data cloud -- that is extrapolation, and this")
    print("    function has already shown what extrapolation costs (y = -215). ***")

# exploit has no exploration term, so watch for it stalling on the incumbent.
d_inc = np.linalg.norm(x_next - X[np.argmax(y)])
print(f"\ndistance from the incumbent: {d_inc:.4f}")
if d_inc < 0.01:
    print("*** exploit is proposing essentially the incumbent again -- it has")
    print("    stopped making progress. Switch to ucb with a small kappa. ***")

print("nearest 3 observations to x_next:")
for i in np.argsort(np.linalg.norm(X - x_next, axis=1))[:3]:
    print(f"  dist={np.linalg.norm(X[i] - x_next):.4f}  y={y[i]:+9.3f}  X={np.round(X[i], 3)}")

## Visualise the GP and acquisition function via 1D slices

Each panel holds the other three dimensions fixed at the current best observed point and sweeps one dimension. Dotted line is the fixed centre, dashed red is the proposed `x_next`.

Because the acquisition is `exploit`, the green acquisition curve is the posterior mean itself, so it tracks the blue GP mean exactly. That's expected. It also makes the reason for the choice visible: the mean has a clear interior optimum on each axis, while the *uncertainty* band widens toward the far ends. Any acquisition weighting that band heavily follows it outward — which is what the rejected configs do.

This is a *partial* view: it shows what the GP believes along each axis near the best point, not interactions between dimensions.

In [ ]:
# exploit_acquisition is just the posterior mean, so the green acquisition curve
# below traces the GP mean itself -- that is expected, not a plotting bug.
plot_nd_slices(
    X, y, bounds, gp,
    acquisition_fn=exploit_acquisition,
    x_next=x_next,
    acq_kwargs={"maximize": True},
)
plt.show()

## Sanity-check the surrogate model: leave-one-out calibration

Refits the GP once per observation, leaving it out, and predicts it from the rest. At D=4 this is the main way to judge the surrogate, since the fitted surface can't be inspected directly.

Read the error bars rather than the correlation. The outlier is the point to watch: it should be badly mispredicted when held out (the rest of the data gives no reason to expect -215), and that is informative rather than alarming -- but it does mean the model's behaviour in that region is an extrapolation resting on a single measurement.

In [ ]:
with fitting("leave-one-out calibration"):
    pred_mean, pred_std = loo_predictions(X, y, bounds,
                                          gp_kwargs={"n_restarts_optimizer": 20})

plot_loo_calibration(y, pred_mean, pred_std)
plt.show()

within = np.abs(y - pred_mean) <= 1.96 * pred_std
print(f"points inside their own 95% LOO interval: {within.sum()}/{len(y)}")

# NOTE: this LOO runs on the MODELLING set, so the week-2 outlier is not in it.
# The old version asked "is the worst-predicted point the collected outlier?" by
# testing `worst == n_initial`; with the outlier excluded that index is week 3,
# so the question no longer means anything. Report which point it is instead.
worst = int(np.argmax(np.abs(y - pred_mean)))
src = "initial batch" if worst < n_initial else f"collected #{worst - n_initial + 1}"
print(f"worst-predicted point: index {worst} ({src}) -> true {y[worst]:.4g},"
      f" predicted {pred_mean[worst]:.4g} (std {pred_std[worst]:.4g})")
print("(the excluded week-2 outlier is absent by construction, so this measures")
print(" the surrogate actually in use rather than the one distorted by it)")

## Iteration diagnostics

`compute_iteration_diagnostics` replays the ordered `X`/`y` to reconstruct what the acquisition value, GP hyperparameters, and domain-wide uncertainty were at each past proposal -- no persisted log involved.

It applies one acquisition setting to the whole history, taken from `ACQ`/`ACQ_KWARGS` so it can't drift from what the proposal used. The single recorded observation was proposed under the old notebook's `pi`/`xi=0.01`, not `exploit`, so its replayed acquisition value describes a decision that was never made that way. Treat that column as meaningless until the history is `exploit` throughout.

One convenient property of replaying as `exploit`: the acquisition value is just the posterior mean at the point *before* it was evaluated, so it reads as "what did the model expect there", which is interpretable however that point was chosen. Compare it against the `y` actually observed to see how badly the model was surprised.

It also assumes row order is true chronological order. `plot_bo_diagnostics` is skipped (it hard-codes a 2D scatter panel and this is 4D); the trend plots below are dimension-agnostic and gated on having a few completed iterations.

In [ ]:
# The old `if len(new_y) == 0` guard is gone: new_y is populated above, so that
# branch was unreachable.
#
# The count must come from the MODELLING set, not len(new_y). The replay only
# sees the rows actually fitted, so with the week-2 outlier excluded there are
# len(y) - n_initial iterations, not len(new_y). The old code used len(new_y)
# and would have over-reported by one and mis-gated the trend plots.
n_replayed = len(y) - n_initial

with fitting("iteration-diagnostics replay"):
    history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial,
                                            acquisition=ACQ, maximize=True, **ACQ_KWARGS)

print("Best y so far:", np.nanmax(history["y"]))
print(f"completed iterations in the modelling set: {n_replayed}"
      f"  ({len(new_y)} collected, {len(new_y) - n_replayed} excluded from the fit)")

if n_replayed >= 3:
    for plot_fn in (plot_convergence, plot_acquisition_decay,
                    plot_uncertainty_shrinkage, plot_step_distance):
        plot_fn(history)
        plt.show()
else:
    print(f"\nOnly {n_replayed} replayed iteration(s) -- need at least 3 before the")
    print("trend plots say anything. Skipping them; the raw fields are below.")

## Raw diagnostic fields

In [ ]:
print("y:", np.round(history["y"], 4))
print("\niteration:", history["iteration"])
print("\nacq_value (NaN = initial batch; see the caveat above):", history["acq_value"])
print("\nlength_scale:", history["length_scale"])
print("\ndomain_mean_std:", history["domain_mean_std"])

In [13]:
# The proposal as a hyphen-separated string, for submission.
print("-".join(f"{v:.6f}" for v in x_next))

# Full precision as well. 6 dp is fine to submit, but paste THIS into next
# week's new_X: a 6-dp copy of THIS function's previous proposal rounded 4e-7
# outside its own upper bound and tripped validate_bounds_consistency.
print("\nfull precision (use for next week's new_X):")
print("-".join(repr(float(v)) for v in x_next))

0.393738-0.377868-0.388294-0.418940

full precision (use for next week's new_X):
0.3937384985978247-0.3778680410648952-0.3882939829422201-0.41894036092135334
